1. Cấu hình và môi trường

Catalogue draft 39 món. Mặc định offline, dùng evidence đã đóng băng trong config; raw metadata trong datasets được bỏ qua bằng Git. REFRESH_WEB chỉ lấy metadata mới, không tự chọn ảnh hay thay nội dung đã biên tập. Không gọi API GPT. Run All không cần nhập/chỉnh dữ liệu giữa các cell.

In [ ]:
from pathlib import Path
import sys, json, platform
import pandas as pd
here = Path.cwd().resolve()
DATA_PREPARATION_DIR = next((p if p.name == 'data-preparation' else p / 'data-preparation' for p in [here, *here.parents] if (p if p.name == 'data-preparation' else p / 'data-preparation').is_dir()), None)
if DATA_PREPARATION_DIR is None:
    raise RuntimeError('Mở kernel trong repo hoặc đặt DATA_PREPARATION_DIR')
sys.path.insert(0, str(DATA_PREPARATION_DIR / 'scripts'))
import editorial_catalogue as ec
import data_contract as dc
OPTIONS = globals().get('NOTEBOOK_OPTIONS', {})
REFRESH_WEB = OPTIONS.get('refresh', False)  # Đổi True khi chủ động lấy metadata mới
OFFLINE = OPTIONS.get('offline', not REFRESH_WEB)
if REFRESH_WEB and OFFLINE:
    raise ValueError('Refresh không thể chạy trong chế độ offline')
OUTPUT_DIR = DATA_PREPARATION_DIR / 'snapshots/editorial-v0.2.0'
REPORTS_DIR = DATA_PREPARATION_DIR / 'datasets/editorial-catalogue/reports'
print({'python': platform.python_version(), 'pandas': pd.__version__, 'offline': OFFLINE, 'refresh': REFRESH_WEB, 'output': str(OUTPUT_DIR)})


2. Khám phá nguồn

Đọc snapshot nguồn, checksum và mapping owner. Giá/metadata menu vẫn nằm theo từng offering. rawCaptures là snapshot lần khảo sát web; nếu còn file trên máy thì checksum phải khớp, clone mới dùng evidence trích xuất đã theo dõi Git.

In [ ]:
bases, config, evidence, images = ec.load_inputs()
source_table = pd.DataFrame(bases)
print('Món gốc:', source_table.shape, 'Cột:', source_table.columns.tolist())
display(source_table[['id', 'name', 'version', 'menuItemCount', 'sourceNameCount']])
mapping = pd.DataFrame(ec.read_csv(DATA_PREPARATION_DIR / 'snapshots/source_to_base_mapping.csv'))
display(mapping['status'].value_counts().rename('rows'))
print(config['snapshotChecksums'])
display(pd.DataFrame(evidence['sources']))
print(evidence['retrievalSummary'])
print('Raw captures có trên máy:', sum((ec.CACHE / x['file']).exists() for x in evidence['rawCaptures']), '/', len(evidence['rawCaptures']))


3. Bảng biên tập từng UUID

Các nhãn được viết riêng theo phạm vi món gốc; GPT chỉ là đề xuất nguồn để đối chiếu. Không ghép đặc tính của những offering khác nhau thành đặc tính chung. field_evidence.csv xuất tham chiếu chính xác từng trường; fieldSources trong contract chỉ lưu được một nguồn cho mỗi nhóm thông tin.

In [ ]:
editorial = pd.DataFrame(config['dishes'])
display(editorial[['id', 'name', 'description', 'aliases', 'cuisineCodes', 'categoryCodes', 'mealSlots', 'origin', 'temperature']])
proofs = [dict(dishId=r['id'], name=r['name'], field=f, **proof) for r in config['dishes'] for f, proof in r['evidence'].items()]
display(pd.DataFrame(proofs))
proposals = [dict(dishName=b['name'], **p) for b in bases for p in b['sourceProfiles']]
print('Đề xuất GPT theo tên nguồn, giữ draft:', len(proposals))
display(pd.DataFrame(proposals).head(10))


4. Nguồn ảnh và license

Lưu file URL, file page, tác giả, license/link, dòng ghi công, revision và ngày kiểm. Metadata không thay kiểm nội dung ảnh. Wikimedia đã chặn xem ảnh bằng 403/429; ứng viên chưa kiểm hình ảnh không được chọn artwork. Ảnh menu có quyền unknown không được tái sử dụng. Refresh tuần tự, tối đa 3 URL mỗi lần, dừng ngay 403/429 và không tự retry; kết quả mới chỉ nằm trong raw cache để biên tập sau.

In [ ]:
if REFRESH_WEB:
    journal = ec.refresh_metadata(evidence['refreshRequests'][:3])
    display(pd.DataFrame(journal))
else:
    print('Dùng evidence/cache: 0 network calls')
print(images['visualInspection'])
image_table = pd.DataFrame([dict(dishId=r['dishId'], dishName=r['dishName'], searchStatus=r['searchStatus'], **c) for r in images['dishes'] for c in r['candidates']])
display(image_table[['dishName', 'title', 'author', 'license', 'licenseUrl', 'pageUrl', 'revision', 'status', 'reason']])
display(pd.DataFrame([dict(name=r['dishName'], searchStatus=r['searchStatus'], reason=r['reason'], candidates=len(r['candidates'])) for r in images['dishes']]))


5. Giá tham chiếu theo menu/quán

Giá của snapshot, không phải giá hiện tại hay giá chung món gốc. Không tự quy thành giá/người; chưa có khẩu phần nên giữ menu_item_unspecified. Các entity nơi bán/lịch/coverage không được nhập từ quan sát chưa xác minh này.

In [ ]:
bundle, prices, image_rows, field_evidence = ec.build(bases, config, evidence, images)
price_table = pd.DataFrame(prices)
print('Giá tham chiếu:', price_table.shape, 'Chi nhánh:', price_table['venueSurveyId'].nunique())
display(price_table[['dishName', 'menuName', 'venueName', 'priceRaw', 'priceValue', 'currency', 'unit', 'checkedAt', 'sourceUrl']])
display(price_table[['currency', 'unit', 'status']].value_counts().rename('rows'))


6. Kiểm tra ID, nguồn, unknown và contract

Mapping 216/2/1, UUID món không đổi, version món tăng từ 1 lên 2. Kiểm từng giá trị giá/chi nhánh/URL với nguồn. Artwork chỉ được nhận khi đủ kiểm nội dung và license; không gắn allergy/ingredientTags từ tên. Tất cả bản ghi vẫn draft, validUntil/reviewer chưa biết.

In [ ]:
assert len(bundle['entities']['dishes']) == 39
assert {r['id'] for r in bases} == {r['id'] for r in bundle['entities']['dishes']}
assert dict(mapping['status'].value_counts()) == {'mapped': 216, 'needs_review': 2, 'excluded': 1}
assert all(r['version'] == 2 and r['status'] == 'draft' and r['reviewedBy'] is None and r['ingredientTags'] is None for r in bundle['entities']['dishes'])
ec.validate_prices(bases, prices)
errors = dc.validate(bundle)
assert not errors, errors
stats = ec.quality(bundle, prices, image_rows)
print(stats)
display(pd.DataFrame(bundle['entities']['dishes'])[['name', 'description', 'cuisineIds', 'categoryIds', 'mealSlots', 'temperature', 'artwork']])
print({k: len(v) for k, v in bundle['entities'].items()})


7. Xuất và đọc lại

Tạo folder tự động, giữ snapshot trước. JSON catalogue và CSV bundle tương đương; các bảng nguồn/ảnh/giá là sidecar nghiên cứu. Manifest băm artifact; chạy hai lần offline có cùng checksum. Source notebook không lưu kết quả; Python runner lưu notebook đã chạy trong datasets/reports (chưa tuyên bố kiểm trực tiếp Jupyter kernel).

In [ ]:
stats = ec.export(bundle, prices, image_rows, field_evidence, config, OUTPUT_DIR)
assert dc.read_csv_bundle(OUTPUT_DIR / 'csv') == bundle
assert json.loads((OUTPUT_DIR / 'catalogue.json').read_text()) == bundle
price_readback = ec.read_csv(OUTPUT_DIR / 'price_references.csv')
assert len(price_readback) == len(prices) == 247
for before, after in zip(prices, price_readback):
    assert after == {k: '' if v is None else str(v) for k, v in before.items()}
for path in OUTPUT_DIR.rglob('*.csv'):
    assert path.read_bytes().startswith(b'\xef\xbb\xbf')
print(OUTPUT_DIR)
display(pd.DataFrame(bundle['entities']['dishes']).head())
print((OUTPUT_DIR / 'QUALITY_REPORT.md').read_text())
display(pd.DataFrame.from_dict(json.loads((OUTPUT_DIR / 'manifest.json').read_text())['files'], orient='index'))
